

## Load NAB time series

In [29]:
## Imports
import pandas as pd
import numpy as np

In [30]:
import os

# Path to Numeta Anomoly Benchmark (NAB) data folder
base_path = "../data/NAB-master/data/realKnownCause/"

# Load one example file
file_path = os.path.join(base_path, "ambient_temperature_system_failure.csv")
df = pd.read_csv(file_path)

df.head()


,timestamp,value
0,2013-07-04 00:00:00,69.880835
1,2013-07-04 01:00:00,71.220227
2,2013-07-04 02:00:00,70.877805
3,2013-07-04 03:00:00,68.959400
4,2013-07-04 04:00:00,69.283551


## Load Anomaly Labels (JSON)

In [31]:
import json  # to work with JSON files

# Tell Python where the labels file is
label_path = "../data/NAB-master/labels/combined_labels.json"

# Open the file and read its contents
with open(label_path, "r") as label_file:      # "r" = read mode, label_file = name for the opened file
    labels = json.load(label_file)             # turn the JSON into a Python dictionary called 'labels'

# 3. Shows the first 10 dataset keys names to confirm it loaded correctly
list(labels.keys())[:10]


['artificialNoAnomaly/art_daily_no_noise.csv',
 'artificialNoAnomaly/art_daily_perfect_square_wave.csv',
 'artificialNoAnomaly/art_daily_small_noise.csv',
 'artificialNoAnomaly/art_flatline.csv',
 'artificialNoAnomaly/art_noisy.csv',
 'artificialWithAnomaly/art_daily_flatmiddle.csv',
 'artificialWithAnomaly/art_daily_jumpsdown.csv',
 'artificialWithAnomaly/art_daily_jumpsup.csv',
 'artificialWithAnomaly/art_daily_nojump.csv',
 'artificialWithAnomaly/art_increase_spike_density.csv']

## Extract Labels for Selected Dataset

In [32]:
# Select the correct dataset key from the labels dictionary
dataset_key = "realKnownCause/ambient_temperature_system_failure.csv"

# Extract the list of anomaly timestamps for this dataset
label_times = labels[dataset_key]

# Display the timestamps
label_times

['2013-12-22 20:00:00', '2014-04-13 09:00:00']

## Add Labels to DataFrame

In [33]:
# Make sure the timestamp column is in datetime format
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)

# Converting the label timestamps into datetime objects 
label_times_dt = pd.to_datetime(label_times)

# Create a new column: 1 if the timestamp is an anomaly, 0 otherwise
df["is_anomaly"] = df["timestamp"].isin(label_times_dt).astype(int)

# See the updated dataframe
df.head()


,timestamp,value,is_anomaly
0,2013-07-04 00:00:00,69.880835,0
1,2013-07-04 01:00:00,71.220227,0
2,2013-07-04 02:00:00,70.877805,0
3,2013-07-04 03:00:00,68.959400,0
4,2013-07-04 04:00:00,69.283551,0


In [34]:
df["is_anomaly"].value_counts()

is_anomaly
0    7265
1       2
Name: count, dtype: int64

## Step 2 – Standardise core columns (`time`, `value`, `is_anomaly`, `case_study`)

In [35]:
# Start from the dataframe we already built:
# df has columns: timestamp, value, is_anomaly

# 1. Rename timestamp -> time
df = df.rename(columns={"timestamp": "time"})

# 2. Add a case_study identifier
df["case_study"] = "ambient"

# Quick check
df.head()


,time,value,is_anomaly,case_study
0,2013-07-04 00:00:00,69.880835,0,ambient
1,2013-07-04 01:00:00,71.220227,0,ambient
2,2013-07-04 02:00:00,70.877805,0,ambient
3,2013-07-04 03:00:00,68.959400,0,ambient
4,2013-07-04 04:00:00,69.283551,0,ambient


## Step 3 – Convert `value` from Fahrenheit to Celsius 


In [36]:
# Convert the main `value` column from Fahrenheit to Celsius in place
df["value"] = (df["value"] - 32.0) * 5.0 / 9.0

# Quick sanity check
df[["time", "value", "is_anomaly", "case_study"]].head()
#df["value"].describe()


,time,value,is_anomaly,case_study
0,2013-07-04 00:00:00,21.044908,0,ambient
1,2013-07-04 01:00:00,21.789015,0,ambient
2,2013-07-04 02:00:00,21.598781,0,ambient
3,2013-07-04 03:00:00,20.533000,0,ambient
4,2013-07-04 04:00:00,20.713084,0,ambient


In [38]:
df["value"].describe()

count    7267.000000
mean       21.801352
std         2.359727
min        14.143559
25%        20.205228
50%        22.143607
75%        23.572754
max        30.124007
Name: value, dtype: float64